# 03 - Feature engineering e dataset de modelagem

## Definição do alvo

- `is_low_review = 1` quando `review_score <= 2`.
- `is_low_review = 0` quando `review_score >= 3`.

Uso de negocio: priorizar pedidos com maior risco de experiencia ruim para investigação operacional, comunicação proativa ou melhoria do processo logístico.

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from olist.config import INTERIM_DATA_DIR, PROCESSED_DATA_DIR, ensure_project_dirs
from olist.data import build_order_level_dataset, load_all
from olist.features import prepare_modeling_dataset

ensure_project_dirs()

In [ ]:
parquet_path = INTERIM_DATA_DIR / 'order_level_dataset.parquet'
csv_path = INTERIM_DATA_DIR / 'order_level_dataset.csv'

if parquet_path.exists():
    order_level = pd.read_parquet(parquet_path)
elif csv_path.exists():
    order_level = pd.read_csv(csv_path, parse_dates=[
        'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
        'order_delivered_customer_date', 'order_estimated_delivery_date'
    ])
else:
    tables = load_all(include_geolocation=False)
    order_level = build_order_level_dataset(tables)

modeling_df = prepare_modeling_dataset(order_level)
print(modeling_df.shape)
display(modeling_df.head())

## Selecionando variáveis

O conjunto abaixo usa informações disponiveis no pedido, pagamento, produto, cliente e entrega. 
O Objetivo aqui é um modelo pré-compra, então as variáveis `delivery_time_days`, `delay_days` e `is_late` não foram selecionadas, pois no cenário operacional real, elas só vão existir com a finalização da entrega. E o objetivo é prever a possibilidade de falhas antes de ocorrerem.

- Dado um pedido recém realizado, consigo antecipar risco de insatisfação antes da entrega acontecer?

In [ ]:
candidate_features = [
    # contexto
    'customer_state',
    'dominant_category',

    # tempo
    'purchase_month',
    'purchase_dayofweek',
    'purchase_hour',
    'is_weekend_purchase',

    # complexidade
    'order_items',
    'unique_products',
    'unique_sellers',

    # financeiro
    'products_value',
    'freight_value',
    'freight_ratio',
    'total_order_value',
    'payment_installments',
    'payment_methods',

    # fÃ­sico
    'total_weight_g',
    'total_volume_cm3',

    # operacional inicial
    'approval_time_hours'
]
id_cols = ['order_id', 'customer_id', 'customer_unique_id']
target_cols = ['review_score', 'is_low_review']
available_features = [col for col in candidate_features if col in modeling_df.columns]

modeling_dataset = modeling_df[id_cols + target_cols + available_features].copy()

display(modeling_dataset.head())
print('Target rate:', modeling_dataset['is_low_review'].mean().round(4))

###### 
- Essas features foram escolhidas porque análises anteriores sugeriram relação plausí­vel com experiència ruim
- Variáveis pós-entrega foram removidas para evitar data leakage e permitir um modelo realmente preventivo.

In [ ]:
missing_summary = (
    modeling_dataset.isna().mean()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'index': 'column', 0: 'missing_rate'})
)
display(missing_summary.head(20))

feature_dictionary = pd.DataFrame({
    'feature': available_features,
    'role': ['categorical' if modeling_dataset[col].dtype == 'object' or str(modeling_dataset[col].dtype).startswith('string') else 'numeric' for col in available_features],
    'business_meaning': [
        'Variavel usada para explicar ou prever baixa satisfacao no nivel do pedido.'
        for _ in available_features
    ],
})
display(feature_dictionary)

In [ ]:
processed_csv = PROCESSED_DATA_DIR / 'modeling_dataset.csv'
processed_parquet = PROCESSED_DATA_DIR / 'modeling_dataset.parquet'
dictionary_csv = PROCESSED_DATA_DIR / 'feature_dictionary.csv'

modeling_dataset.to_csv(processed_csv, index=False)
feature_dictionary.to_csv(dictionary_csv, index=False)

try:
    modeling_dataset.to_parquet(processed_parquet, index=False)
    print(f'Dataset salvo em: {processed_parquet}')
except Exception as exc:
    print(f'Parquet indisponivel ({exc}). CSV salvo em: {processed_csv}')

print(f'Dicionario salvo em: {dictionary_csv}')